In [13]:
#!/usr/bin/env python3

import cv2
import pytesseract
import numpy as np

def fix_rotation_and_extract(image_path):
    """Fix 90-degree rotation and extract text"""
    
    print("Loading image...")
    image = cv2.imread(image_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    print(f"Original dimensions: {gray.shape[1]} x {gray.shape[0]}")

    
    # Now try OCR on the properly oriented image
    print("\nRunning OCR on corrected image...")
    
    # Based on your diagnostic, "Single Column" mode worked best (77.2% confidence)
    best_config = '--oem 3 --psm 4'  # Single column mode
    
    try:
        # Extract text
        extracted_text = pytesseract.image_to_string(gray, config=best_config)
        
        # Get confidence data
        data = pytesseract.image_to_data(gray, config=best_config, output_type='dict')
        confidences = [int(conf) for conf in data['conf'] if int(conf) > 0]
        avg_confidence = np.mean(confidences) if confidences else 0
        
        print(f"\n✅ OCR RESULTS:")
        print(f"   Confidence: {avg_confidence:.1f}%")
        print(f"   Text length: {len(extracted_text)} characters")
        
        # Fix the f-string backslash issue
        lines = extracted_text.split('\n')
        non_empty_lines = len([line for line in lines if line.strip()])
        print(f"   Non-empty lines: {non_empty_lines}")
        
        # Show first part of extracted text
        print(f"\n📄 EXTRACTED TEXT (first 500 chars):")
        print("=" * 60)
        print(extracted_text[:500])
        if len(extracted_text) > 500:
            print("... (truncated)")
        print("=" * 60)
        
        # Save full text to file
        with open('extracted_text.txt', 'w', encoding='utf-8') as f:
            f.write(extracted_text)
        print(f"\n💾 Full extracted text saved to 'extracted_text.txt'")
        
        return extracted_text, avg_confidence
        
    except Exception as e:
        print(f"❌ OCR Error: {e}")
        return None, 0

def extract_sales_data_from_text(text):
    """Parse specific sales report fields from extracted text"""
    
    import re
    
    print("\n🔍 PARSING SALES REPORT DATA:")
    
    # Extract key fields using regex patterns
    patterns = {
        'Date': r'Date\s*:?\s*(\d{1,2}/\d{1,2}/\d{2,4})',
        'Showroom': r'Showroom\s*:?\s*([A-Z\s\(\)0-9]+)',
        'Names': r'\\b([A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*)\\b',
        'Dollar Amounts': r'\\$\\s*([0-9,]+\\.?[0-9]*)',
        'Total Sales': r'Total.*?\\$\\s*([0-9,]+\\.?[0-9]*)',
    }
    
    extracted_fields = {}
    
    for field_name, pattern in patterns.items():
        matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
        if matches:
            extracted_fields[field_name] = matches
            print(f"   {field_name}: {len(matches)} found")
            if field_name in ['Date', 'Showroom']:
                print(f"      → {matches[:3]}")  # Show first few
        else:
            print(f"   {field_name}: None found")
    
    return extracted_fields

if __name__ == "__main__":
    # Your image path
    image_path = input()
    
    try:
        # Fix rotation and extract text
        extracted_text, confidence = fix_rotation_and_extract(image_path)
        
        if extracted_text and confidence > 50:
            print(f"\n🎉 SUCCESS! OCR confidence: {confidence:.1f}%")
            
            # Parse sales report data
            sales_data = extract_sales_data_from_text(extracted_text)
            
            if sales_data:
                print(f"\n📊 EXTRACTED SALES DATA:")
                for field, values in sales_data.items():
                    if values:
                        print(f"   {field}: {values[:5]}")  # Show first 5 items
            
        else:
            print(f"\n⚠️  OCR confidence too low ({confidence:.1f}%)")
            print("The rotated image might need additional preprocessing.")
            print("Check 'rotated_document.jpg' to verify the rotation is correct.")
            
    except FileNotFoundError:
        print(f"❌ Error: Could not find '{image_path}'")
        print("Please update the image_path variable with your actual file path.")
    except Exception as e:
        print(f"❌ Unexpected error: {e}")

Loading image...
Original dimensions: 3264 x 2320

Running OCR on corrected image...

✅ OCR RESULTS:
   Confidence: 58.2%
   Text length: 1338 characters
   Non-empty lines: 39

📄 EXTRACTED TEXT (first 500 chars):
Showroom : NOVA ( TRADEHUB 21 )

Date : 1/08/2025 (FRIDA’

13

Ps [ken 1-302765 | $1,513.76 _| 87,650.00 [Wd
ae ee a eZ a:
Ps | sonn [4-302767 | $1,847.00 [$2,013.23 47 dtl 2.013.23 | Fd
[Ps | zoewe {1-sozres | se.146.r@ [LF s.650.007] PEST [$3,680.00 |__| rennet
Ls | Zoewe —["1-302769 {$2,000.00 ft 82,180.00 4 _| $2,180.00 |__APET_| Se
Foun [0771 [—st808.80—[-] $1,750.00 A | ne $1,750.00 w
pe [| tom | 1-302772 | $1,720.00 [aya soraoorAT TC $974.00 | RT
["* [sonw [4302773 | $2,499.00_| _—«|:-Sa7
... (truncated)

💾 Full extracted text saved to 'extracted_text.txt'

🎉 SUCCESS! OCR confidence: 58.2%

🔍 PARSING SALES REPORT DATA:
   Date: 1 found
      → ['1/08/2025']
   Showroom: 2 found
      → ['NOVA ( TRADEHUB 21 )\n\nDate ', '\n']
   Names: None found
   Dollar Amounts: None